In [1]:
import os

from google import genai
from google.genai import types
from google.genai import errors

# Set GEMINI_API_KEY and STOCK_TICKER in the environment before running this notebook.
symbol = os.environ.get("STOCK_TICKER", "").strip().upper()
if not symbol:
    raise RuntimeError("Set STOCK_TICKER before running enterprise.ipynb")

api_key = os.getenv("GEMINI_API_KEY", "").strip()
if not api_key:
    raise RuntimeError("Set GEMINI_API_KEY before running enterprise.ipynb")

client = genai.Client(api_key=api_key)

# Ordered list of models for fallback to handle rate limits
fallback_models = [
    "gemini-3.8-flash",
    "gemini-3.0-flash",       # Standardized "Gemini 3 Flash"
    "gemini-2.5-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-2.5-flash-lite"
]

description_response = None
successful_model = None

for model_name in fallback_models:
    try:
        print(f"Attempting to generate description using {model_name}...")
        description_response = client.models.generate_content(
            model=model_name,
            contents=(
                f"Describe the company identified by stock ticker {symbol}. "
                "If the ticker is unknown or ambiguous, say so instead of guessing."
            ),
            config=types.GenerateContentConfig(
                system_instruction="Answer professionally in no more than 200 words. Describe the company's business concisely, without bullet points. You may include historical facts.",
                temperature=0.2,
            ),
        )
        # If the request succeeds, record the model and break out of the loop
        successful_model = model_name
        break
        
    except errors.APIError as e:
        # Check if the error is due to rate limiting (HTTP 429)
        if e.code == 429:
            print(f"  -> Rate limit exceeded for {model_name}. Falling back to next model...\n")
            continue
        else:
            # If it's a different API error (e.g., 400 Bad Request), you may want to fail immediately
            print(f"  -> API Error with {model_name}: {e}\n")
            continue
    except Exception as e:
        # Catch unexpected errors (like network timeouts) and continue to the next model
        print(f"  -> Unexpected error with {model_name}: {e}\n")
        continue

# Check if we exhausted all models without getting a response
if not description_response:
    raise RuntimeError(f"Failed to retrieve a description for {symbol} after exhausting all fallback models.")

description = (description_response.text or "").strip()
if not description:
    raise RuntimeError(f"Gemini returned an empty description for {symbol} using {successful_model}")

print(f"\nCompany description: {symbol} (Generated via {successful_model})\n")
print(description)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Attempting to generate description using gemini-3.8-flash...


  -> Rate limit exceeded for gemini-3.8-flash. Falling back to next model...

Attempting to generate description using gemini-3.0-flash...
  -> API Error with gemini-3.0-flash: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3.0-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

Attempting to generate description using gemini-2.5-flash...


  -> API Error with gemini-2.5-flash: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}

Attempting to generate description using gemini-3.5-flash-lite...


  -> API Error with gemini-3.5-flash-lite: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

Attempting to generate description using gemini-3.1-flash-lite...



Company description: CSCO (Generated via gemini-3.1-flash-lite)

Cisco Systems, Inc. (CSCO) is a global technology leader specializing in networking hardware, software, and telecommunications equipment. Founded in 1984 by Stanford University computer scientists Leonard Bosack and Sandy Lerner, the company gained early prominence by developing the multi-protocol router, which allowed disparate computer networks to communicate effectively. This innovation became a foundational pillar of the modern internet.

Today, Cisco’s business encompasses a broad portfolio of enterprise solutions, including routing, switching, cybersecurity, collaboration tools, and cloud-managed services. The company has strategically transitioned from a hardware-centric model toward a recurring revenue structure driven by software subscriptions and services. By integrating artificial intelligence and advanced analytics into its infrastructure, Cisco supports the digital transformation of businesses, governments, 